# How the code works
I’ve learned that Databricks allows you to create notebooks that function similarly to functions—meaning they accept certain values and perform specific tasks—but they’re based on notebooks rather than separate files, as is the case when creating larger, more complex projects in Java.

In [0]:
import yaml

config_path = "/Volumes/dbt_bara_project/table_info/configurations_fales/config.yml"

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)
print(config)

{'paths': {'raw_source_base': '/Volumes/raw_data/dbt_bara_data/', 'checkpoint_base': '/Volumes/dbt_bara_project/table_info/checkpoint/', 'schema_base': '/Volumes/dbt_bara_project/table_info/schema_base/'}, 'systems': {'crm': {'target_catalog': 'dbt_bara_project', 'target_schema': 'staging'}, 'erp': {'target_catalog': 'dbt_bara_project', 'target_schema': 'staging'}}, 'allowd_systems': ['crm', 'erp']}


In [0]:
table_name = dbutils.widgets.get("table_name")
system_key = table_name.split("_")[0]

if system_key not in config['allowd_systems']:
    raise ValueError(f"Unsupported system: {system_key}. Check the config.yaml file.")

sys_config = config['systems'][system_key]
paths = config['paths']

print(table_name)
print(paths)

In [0]:
full_raw_path = f"{paths['raw_source_base']}{table_name}"
full_schema_path = f"{paths['schema_base']}lnd_{table_name}"
full_checkpoint_path = f"{paths['checkpoint_base']}lnd_{table_name}"
target_table_name = f"{sys_config['target_catalog']}.{sys_config['target_schema']}.lnd_{table_name}"
print(full_raw_path)

In [0]:
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    # I introduced this option so that the cat would always run when it received a message from 
    # an external server about a change in the status of a given database. However, given the 
    # local nature of my files, this option is unnecessary in this case.

    #.option("cloudFiles.useManagedFileEvents","true")
    
    # This option is necessary to avoid the error "Cannot merge schema with different case 
    # sensitivity" when the schema is updated.
    .option("cloudFiles.inferColumnTypes", "true")
    
    # Due to the fact that in my code you tried to automate operations so as not to write 
    # rigid structures, but to automatically detect column types, I am adding another column 
    # here to which all errors will be written so that I can detect and fix them.
    .option("cloudFiles.rescuedDataColumn", "_error_record")
    
    # Due to the fact that mainly historical files will be stored here, this option blocks 
    # the creation of new columns to prevent these files from being corrupted.
    .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
    
    # Download only files with a specific format so that if a configuration file or other 
    # file is accidentally inserted, it will not destroy the table. And enable the insertion 
    # of various configuration files.
    .option("pathGlobFilter", "*.csv")
    .option("cloudFiles.schemaLocation", f"{full_schema_path}")
    .load(f"{full_raw_path}"))

(df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", full_checkpoint_path)
    .toTable(target_table_name))